# SmartHandover — Day 1: VADER Baseline

Corre o classificador VADER (lexico) sobre o dataset MELD e calcula metricas de avaliacao.

**Toggle:** muda `USE_MOCK = True` para usar dados dummy (sem download).

In [ ]:
%pip install vaderSentiment pandas scikit-learn tqdm matplotlib datasets librosa soundfile torch

In [ ]:
# === CONFIG ===
USE_MOCK = False  # True = dados dummy, False = MELD real

In [ ]:
import os, sys
import pandas as pd
from tqdm.notebook import tqdm
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
from src.classifiers.vader_classifier import VaderClassifier
TARGET_LABELS = ["anger", "frustration", "sadness", "neutral", "satisfaction"]
TARGET_LABEL2ID = {label: i for i, label in enumerate(TARGET_LABELS)}

## 1. Carregar dados

In [ ]:
def create_mock_data():
    samples = [
        ("I am so angry at you right now!", "anger"), ("This is absolutely unacceptable, I'm furious!", "anger"),
        ("You ruined everything, I hate this!", "anger"), ("Stop doing that, it's driving me crazy!", "anger"),
        ("This is so frustrating, nothing works.", "frustration"), ("I've been waiting for an hour, this is ridiculous.", "frustration"),
        ("Why does this keep happening to me?", "frustration"), ("Every time I try, something goes wrong.", "frustration"),
        ("I feel so sad and lonely today.", "sadness"), ("It breaks my heart to see this.", "sadness"),
        ("Nothing seems to matter anymore.", "sadness"), ("I just want to cry.", "sadness"),
        ("The meeting is at 3pm.", "neutral"), ("Sure, that works for me.", "neutral"),
        ("Let me know when you're ready.", "neutral"), ("Okay.", "neutral"), ("I see.", "neutral"),
        ("This is wonderful, I'm so happy!", "satisfaction"), ("Great job, I really appreciate your help!", "satisfaction"),
        ("Everything worked out perfectly!", "satisfaction"), ("Thank you so much, this is amazing!", "satisfaction"),
    ]
    return pd.DataFrame(samples, columns=["text", "target_emotion"])

def load_meld_as_dataframe():
    from src.data.load_meld import load_meld
    rows = []
    for split in ["train", "validation", "test"]:
        print(f"  Loading MELD split: {split} ...")
        ds = load_meld(split=split, streaming=False)
        for example in ds:
            rows.append({"text": example["text"], "target_emotion": example["target_emotion"], "split": split})
    return pd.DataFrame(rows)

if USE_MOCK:
    print("[INFO] A usar dados MOCK.")
    df = create_mock_data()
else:
    print("[INFO] A carregar MELD real...")
    try:
        df = load_meld_as_dataframe()
    except Exception as e:
        print(f"[WARN] Falha: {e}\n Fallback para mock.")
        df = create_mock_data()

print(f"\nTotal amostras: {len(df)}")
df["target_emotion"].value_counts()

## 2. Correr VADER

In [ ]:
classifier = VaderClassifier()
predictions = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="VADER"):
    result = classifier.predict_and_classify(row["text"])
    predictions.append({"text": row["text"], "true_label": row["target_emotion"], "predicted_class": result["predicted_class"],
                         "vader_pos": result["pos"], "vader_neg": result["neg"], "vader_neu": result["neu"], "vader_compound": result["compound"]})
results_df = pd.DataFrame(predictions)
results_df.head(10)

## 3. Guardar CSV + Metricas

In [ ]:
output_path = os.path.join("..", "data", "processed", "vader_predictions.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
results_df.to_csv(output_path, index=False)
print(f"Guardado em: {output_path}")

from src.evaluation.metrics import compute_metrics, print_metrics
y_true = [TARGET_LABEL2ID[l] for l in results_df["true_label"]]
y_pred = [TARGET_LABEL2ID[l] for l in results_df["predicted_class"]]
metrics = compute_metrics(y_true, y_pred, target_names=TARGET_LABELS)
print_metrics(metrics)

## 4. Distribuicao do compound score por classe

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
for label in TARGET_LABELS:
    subset = results_df[results_df["true_label"] == label]["vader_compound"]
    ax.hist(subset, bins=40, alpha=0.5, label=label, density=True)
ax.set_xlabel("VADER Compound Score"); ax.set_ylabel("Densidade")
ax.set_title("Distribuicao do compound score por classe real"); ax.legend()
ax.axvline(x=-0.3, color="red", linestyle="--", alpha=0.6)
ax.axvline(x=0.1, color="orange", linestyle="--", alpha=0.6)
ax.axvline(x=0.3, color="green", linestyle="--", alpha=0.6)
plt.tight_layout(); plt.show()